# Data Preprocessing — Houses
**Improvements:** Same fixes as flats — `FutureWarning` inplace assignments replaced with direct assignment, safe price parser, duplicate investigation added.

In [ ]:
import numpy as np
import pandas as pd
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
df = pd.read_csv('houses.csv')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# FIX: show duplicate rows before dropping so we can verify they are true duplicates
print(f"Duplicates found: {df.duplicated().sum()}")
df[df.duplicated(keep=False)].sort_values(list(df.columns[:5])).head(10)

In [ ]:
df = df.drop_duplicates().copy()
print(f"Shape after dedup: {df.shape}")
print("Missing values:\n", df.isnull().sum())

In [ ]:
df.drop(columns=['link', 'property_id'], inplace=True)
df.rename(columns={'rate': 'price_per_sqft'}, inplace=True)

In [ ]:
# Clean society names
df['society'] = df['society'].apply(
    lambda name: re.sub(r'\d+(\.\d+)?\s?\u2605', '', str(name)).strip().lower()
)
# Replace literal 'nan' string with 'independent'
df['society'] = df['society'].str.replace('nan', 'independent', regex=False)
print(f"Unique societies: {df['society'].nunique()}")

In [ ]:
# Remove 'Price on Request'
df = df[df['price'] != 'Price on Request'].copy()
print(f"Rows after filter: {len(df)}")

In [ ]:
def treat_price(val):
    parts = val.split(' ')
    if len(parts) < 2:
        return np.nan
    num = float(parts[0])
    unit = parts[1]
    if unit == 'Lac':
        return round(num / 100, 2)
    elif unit == 'Cr':
        return round(num, 2)
    return np.nan

df['price'] = df['price'].apply(treat_price)
df = df[df['price'].notna()].copy()

In [ ]:
df['price_per_sqft'] = (
    df['price_per_sqft']
    .str.split('/').str[0]
    .str.replace('\u20b9', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

In [ ]:
df['bedRoom'] = df['bedRoom'].str.split(' ').str[0]
df = df[df['bedRoom'].notna() & (df['bedRoom'] != 'nan')].copy()
df['bedRoom'] = df['bedRoom'].astype(int)
print(f"Rows after bedRoom cleanup: {len(df)}")

In [ ]:
df['bathroom'] = df['bathroom'].str.split(' ').str[0].astype(int)

In [ ]:
df['balcony'] = df['balcony'].str.split(' ').str[0].str.replace('No', '0')

In [ ]:
# FIX: direct assignment — avoids FutureWarning from inplace on chained loc
df['additionalRoom'] = df['additionalRoom'].fillna('not available')
df['additionalRoom'] = df['additionalRoom'].str.lower()

In [ ]:
# Parse floorNum (noOfFloor) — extract first digit
def parse_floor(val):
    match = re.search(r'\d+', str(val))
    return int(match.group()) if match else np.nan

df.rename(columns={'noOfFloor': 'floorNum'}, inplace=True)
df['floorNum'] = df['floorNum'].apply(parse_floor)

In [ ]:
# FIX: direct assignment
df['facing'] = df['facing'].fillna('NA')

In [ ]:
df['area'] = round((df['price'] * 10000000) / df['price_per_sqft'])
df['property_type'] = 'house'

print(f"Final shape: {df.shape}")
df.info()

In [ ]:
df.to_csv('house_cleaned.csv', index=False)
print("Saved house_cleaned.csv")